# 4. Evanescent diffraction orders

**What happens to the integer orders that our plots leave out?**

[Notebook three](../3_Why_Diffraction_Orders_Appear.ipynb) connected diffraction orders to an integer number of extra phase cycles per grating period relative to the incident wave. Its wave plots showed only candidates with real outgoing directions. Here we follow one of the excluded candidates and construct its field.

The route is **a missing order → its normal wavevector component → a decaying field → a decay length → the transition through grazing**.

Run the cells from top to bottom in a Jupyter kernel with NumPy, Matplotlib, Plotly and ipywidgets. The main explorer in section 4 keeps evanescent orders selectable. Each **small calculation** has its own editable parameters; changing a calculation does not move the explorer controls. Read the prediction, run the cell, and then open the answer.

We retain the earlier geometry: the grating lies at $z=0$, repeats along $x$ with period $d$, and has grooves along $y$. Incidence is from $z>0$; the outgoing side is $z<0$. Angles are measured from $-z$ toward $+x$, and $k_{iy}=k_{my}=0$. Both sides have the same homogeneous, isotropic, lossless medium. $\lambda$ is the wavelength **in that medium**, with lengths in micrometres.

Use $\mathbf k_i$ for the incident wavevector and $\mathbf k_m=(k_{mx},0,k_{mz})$ for diffraction order $m$. Plain $k=2\pi/\lambda$ is the positive scalar wavenumber of the medium; it is distinct from the vector $\mathbf k_m$.

The field examples prescribe **one component with surface amplitude $A=1$**. They show what that component would do if excited. They do not determine which amplitudes a particular grating generates or combine the components into a full scattering solution.


## 1. Keep an order that notebook three could not draw

Periodicity requires

$$k_{mx}=k_{ix}+mG,\qquad G=\frac{2\pi}{d},\qquad m\in\mathbb Z,$$

where $m$ is the diffraction-order number. This is a condition on phase variation **along the surface**. The condition remains meaningful even when the corresponding field has no real outgoing angle. The separation between direction and strength is also discussed in [Ansys, “Grating physics”](https://optics.ansys.com/hc/en-us/articles/360034394354-Grating-projections-in-FDTD-overview).

Use notebook three's starting values:

$$\lambda=d=2\,\mu\mathrm m,\qquad \theta_i=30^\circ,\qquad
k=\pi\,\mathrm{rad}/\mu\mathrm m,\qquad \frac{k_{ix}}k=\frac12.$$

Then

$$\frac{k_{mx}}k=\frac{k_{ix}}k+m\frac{G}{k}
=\frac12+m\frac{\lambda}{d}=\frac12+m.$$

| Order $m$ | $k_{mx}/k$ | Real outgoing direction? |
| --- | --- | --- |
| $-2$ | $-1.5$ | No |
| $-1$ | $-0.5$ | Yes: $-30^\circ$ |
| $0$ | $+0.5$ | Yes: $+30^\circ$ |
| $+1$ | $+1.5$ | No |

The earlier dropdown listed $m=-1$ and $m=0$. We will start by selecting **$m=+1$** instead.

**Predict:** does $m=+1$ fail the integer phase condition? Or does it satisfy that condition but fail a separate condition for a real direction?

<details><summary>Answer</summary>

It satisfies the integer phase condition: $(k_{mx}-k_{ix})d=2\pi$. The difficulty is that $k_{mx}=1.5k$. A real vector of length $k$ cannot have a horizontal component longer than $k$.

</details>

For propagating and grazing orders, $k_{mx}/k=\sin\theta_m$ is the $x$ direction cosine; recall that $\theta_m$ is measured from $-z$. For an evanescent order, $|k_{mx}/k|>1$ remains a valid normalized horizontal wavevector component, but no real angle has that sine. The real-vector triangle from the earlier notebooks therefore cannot represent this evanescent component.


### Small calculation 1: list the candidates before filtering them

Compute the normalized horizontal component and the remaining squared normal component. A positive result permits propagation away from the grating; zero is grazing; a negative result is evanescent. This calculation prints the candidates instead of deleting the evanescent ones.

**Predict and rerun:** keep $\lambda=d$ but change incidence to $0^\circ$. What happens to $m=\pm1$? The small tolerance below treats roundoff at grazing as zero.


In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display, Image

wavelength, d = 2.0, 2.0  # micrometres
theta_i_deg = 30.0
orders = np.arange(-2, 3)
alpha = np.sin(np.deg2rad(theta_i_deg)) + orders*wavelength/d
radicand = 1 - alpha**2  # k_mz**2 / k**2

print(" m    k_mx/k  k_mz²/k²   classification")
for m, a, r in zip(orders, alpha, radicand):
    state = "grazing" if abs(r) <= 1e-12 else ("propagating" if r > 0 else "evanescent")
    print(f"{m:+2d}   {a:+7.3f}   {r:+7.3f}    {state}")


<details><summary>Answer for the starting values and the suggested change</summary>

At $30^\circ$, the normalized horizontal components for $m=-2,-1,0,+1,+2$ are $-1.5,-0.5,+0.5,+1.5,+2.5$. Only $m=-1,0$ are propagating.

At normal incidence, $k_{mx}/k=m$. The zero order propagates, $m=\pm1$ are grazing, and $m=\pm2$ are evanescent. A grazing wave travels along the grating; it does not travel into the outgoing half-space.

</details>


## 2. Complete the normal component

For a field component in the uniform medium, the wave equation requires

$$k_{mx}^2+k_{mz}^2=k^2,
\qquad\text{so}\qquad
k_{mz}^2=k^2-k_{mx}^2.$$

In the outgoing half-space $z<0$, choose the following branches:

| Condition | Normal component $k_{mz}$ | Behaviour with distance from the grating |
| --- | --- | --- |
| $|k_{mx}|<k$ | $-\sqrt{k^2-k_{mx}^2}$ | Phase advances as the wave travels away |
| $|k_{mx}|=k$ | $0$ | Grazing: no normal phase change or decay |
| $|k_{mx}|>k$ | $-i\kappa_m$, with $\kappa_m=\sqrt{k_{mx}^2-k^2}>0$ | Amplitude decays away |

Here $i$ is the imaginary unit, defined by $i^2=-1$. For the last row, $(-i\kappa_m)^2=-\kappa_m^2$, providing the required negative square.

For real propagating or grazing vectors, the equation is also the familiar length condition $|\mathbf k_m|=k$. **Once a component is imaginary, use the algebraic squares shown above.** In vector form this is $\mathbf k_m\cdot\mathbf k_m=k^2$, with no complex conjugation. It is not an ordinary length statement for a complex vector. Replacing $k_{mz}^2$ with $|k_{mz}|^2$ would change the equation; $k$ still denotes the positive wavenumber of the medium.

The normal-component relation and the outgoing branch are developed in [TU Delft, “Angular Spectrum Method,” Eq. (375)](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html#angular-spectrum-method). That source uses propagation toward $+z$; our outgoing side is $-z$, so the signs of the normal components are reversed.

<details><summary>Optional: where does the equation come from?</summary>

The scalar Helmholtz equation in our two-dimensional uniform region is

$$\frac{\partial^2 U}{\partial x^2}+\frac{\partial^2 U}{\partial z^2}+k^2U=0.$$

Try a separated field $U=Ae^{i(k_{mx}x+k_{mz}z)}$. Each derivative in $x$ brings down $ik_{mx}$, so the second derivative is $-k_{mx}^2U$. The corresponding result in $z$ is $-k_{mz}^2U$. Substituting gives

$$(-k_{mx}^2-k_{mz}^2+k^2)U=0.$$

For a nonzero component, the expression in parentheses must vanish. This derivation works with real or complex components; it never introduced a complex conjugate.

</details>


## 3. An imaginary component produces a real, decaying field

Keep the time convention from the earlier notebooks:

$$u_m(x,z,t)=\operatorname{Re}\{U_m(x,z)e^{-i\omega t}\},\qquad
U_m(x,z)=A e^{i(k_{mx}x+k_{mz}z)}.$$

The capital $U_m$ is the time-independent **complex scalar field**. The lower-case $u_m$ is the real, signed field that we plot. For these examples, $A>0$ is the amplitude at the surface and the initial phase is zero.

For an evanescent order, insert $k_{mz}=-i\kappa_m$:

$$e^{ik_{mz}z}=e^{i(-i\kappa_m)z}=e^{\kappa_m z}.$$

Since the outgoing side has $z<0$, this factor gets **smaller** farther below the surface. It is helpful to name the positive distance into that side:

$$s=-z\geq0.$$

The field becomes

$$\boxed{u_m(x,-s,t)=A e^{-\kappa_m s}
\cos(k_{mx}x-\omega t).}$$

There are two separate factors to read:

- $e^{-\kappa_m s}$ is the **amplitude envelope**. It decreases with distance and is independent of time.
- $\cos(k_{mx}x-\omega t)$ oscillates along the surface and in time. At a fixed $x$, its phase is the same at every depth.

This is the meaning of *evanescent in the normal direction*. The field is not identically zero beyond the surface, and it still oscillates. The alternative branch $k_{mz}=+i\kappa_m$ would grow as $s$ increases without limit; it does not describe an isolated grating driving an unbounded outgoing region.

**Predict:** at $x=0$ and $t/T=1/4$, is a zero signed field proof that the amplitude envelope has vanished?


### Small calculation 2: distinguish time from distance

Use the excluded $m=+1$ order from section 1. First evaluate its complex field $U_m$ at three depths. Then multiply by the time factor and take the real part, exactly as in the boxed expression. In Python, `1j` represents $i$, and `np.abs(U_m)` gives the local amplitude.

**Predict and rerun:** change only `tau` from $0$ to $0.25$, then $0.5$. Which printed column should remain unchanged?


In [ ]:
wavelength, d = 2.0, 2.0  # micrometres
theta_i_deg, m = 30.0, 1
tau = 0.0  # t/T; try 0.25, then 0.5
k = 2*np.pi/wavelength
k_ix = k*np.sin(np.deg2rad(theta_i_deg))
k_mx = k_ix + m*2*np.pi/d

if abs(k_mx) <= k:
    print("This calculation needs an evanescent order: choose |k_mx| > k.")
else:
    kappa_m = np.sqrt(k_mx**2-k**2)
    k_mz = -1j*kappa_m
    s = np.array([0.0, 0.25, 0.5])  # positive distance into z < 0
    x = 0.0
    U_m = np.exp(1j*(k_mx*x + k_mz*(-s)))  # complex field; surface amplitude A = 1
    field = np.real(U_m*np.exp(-2j*np.pi*tau))
    print(f"kappa_m = {kappa_m:.3f} per micrometre; k_mz = {k_mz:.3f}")
    print("   s (um)    |U|/A     u/A")
    print(np.round(np.column_stack([s, np.abs(U_m), field]), 3))


<details><summary>Answer</summary>

Here $\kappa_m=\pi\sqrt{1.25}\approx3.512\,\mu\mathrm m^{-1}$. At $s=0,0.25,0.5\,\mu\mathrm m$, the amplitudes are approximately $1,0.416,0.173$.

At $t/T=0$, the signed field at $x=0$ equals those amplitudes. At $t/T=1/4$, it is zero at all three depths, to numerical precision. At $t/T=1/2$, it equals the negatives of those amplitudes. **The amplitude column does not change with time.**

In the spatial plot, other $x$ positions can still have nonzero signed fields when the entire $x=0$ depth cut is zero.

</details>


## 4. Explore the field of the selected order

The explorer starts with $\lambda=d=2\,\mu\mathrm m$, $\theta_i=30^\circ$, and the previously excluded order $m=+1$.

- **Top ruler:** a sample of the integer candidates, $m=-3,\ldots,+3$, placed at $k_{mx}/k$. The shaded band spans $-1$ to $+1$; its edges are grazing. A ring marks the selected order. This ruler is a component axis, not a drawing of the grating.
- **Left plot:** the selected signed field below $z=0$. Red and blue mean opposite signs; the same linear scale, $-1$ to $+1$, is used in every state. The dotted vertical line is $x=0$.
- **Right plot:** the signed field along that dotted line (blue dashed curve), together with its positive and negative amplitude envelope (orange). Its horizontal coordinate is the positive distance $s=-z$.
- **Readout:** the selected order's component and classification. For evanescent orders it also gives the amplitude decay length $\ell=1/\kappa_m$, explained in section 5.

**Try these in order:**

1. Start with $m=+1$. Move **Time t/T**, or press Play. The stripes move along $x$ while their contrast decreases with depth. The orange envelope remains fixed.
2. Set $t/T=0.25$. At $x=0$, the blue depth curve is zero. Compare it with the orange envelope and with other $x$ positions in the field.
3. Select $m=0$ with the same illumination. Its amplitude does not decay, and the signed field generally changes phase with depth.
4. Restore **Notebook 3 example**, then select $m=+2$. Does the field reach as far below the surface?
5. Return to $m=+1$. Change wavelength or period until the classification changes. **The order remains selected**, even when it is evanescent.

**Depth shown** zooms both spatial axes, preserving the physical aspect ratio: the horizontal range extends from minus that depth to plus that depth. It changes the view, not the field. Use a shallow window to inspect rapid decay. The seven displayed order numbers are a sample, not a count of all available orders. The three state-example buttons set normal incidence and the parameters used in section 6; **Notebook 3 example** restores the tilted starting case.


In [ ]:
# Rerunning this cell closes the old explorer before creating its replacement.
if 'evanescent_explorer' in globals():
    evanescent_explorer.close()

import wave_optics.evanescent as evanescent
import wave_optics.evanescent_view as evanescent_view
importlib.reload(evanescent)
importlib.reload(evanescent_view)

evanescent_explorer = evanescent_view.EvanescentOrderExplorer()
display(evanescent_explorer.ui)


## 5. How far does the field extend?

The envelope is $A(s)/A(0)=e^{-\kappa_m s}$. Define the **amplitude decay length**

$$\boxed{\ell_m=\frac{1}{\kappa_m}
=\frac{1}{\sqrt{k_{mx}^2-k^2}}.}$$

At $s=\ell_m$, the amplitude is $e^{-1}\approx0.368$ of its surface value. At $2\ell_m$ it is $e^{-2}\approx0.135$. This is a gradual decay, not a hard boundary.

For the starting $m=+1$ example,

$$\ell_{+1}\approx0.285\,\mu\mathrm m.$$

For $m=+2$ at the same wavelength, period and incidence, $k_{mx}/k=2.5$, so $\kappa_m$ is larger and $\ell_{+2}\approx0.139\,\mu\mathrm m$: this component decays faster. The comparison holds the surface amplitudes equal to isolate the effect of propagation.

A spatial detail can therefore be appreciable near the grating yet contribute very little at a more distant plane. Components just beyond grazing are a useful exception to the phrase “very near”: as $|k_{mx}|$ approaches $k$ from above, $\kappa_m$ approaches zero and the decay length becomes large.

**Experimental connection: which decay does the instrument detect?** A
near-field-sensitive measurement needs a model of the probe's response. If
the calibrated signal follows field amplitude, its decay length is
$\ell_m=1/\kappa_m$. If it follows squared field magnitude instead,

$$\frac{|U(s)|^2}{|U(0)|^2}=e^{-2\kappa_m s},$$

so that signal's $1/e$ length is $1/(2\kappa_m)=\ell_m/2$. These are different
fits to different quantities. Squared amplitude here is not normal transmitted
power; Section 7 explains the distinction. An ordinary far-field order image
does not directly reveal this evanescent decay profile.


### Small calculation 3: one, two, and three decay lengths

Calculate the envelope for $m=+1$, then repeat for $m=+2$. Compare the distances in micrometres with the amplitude ratios.

We also print the **squared amplitude ratio**. It falls twice as fast in the exponent. This quantity is not a calculation of power flowing away from the grating; section 7 explains why that distinction matters.


In [ ]:
wavelength, d = 2.0, 2.0
theta_i_deg, m = 30.0, 1  # repeat with m = 2
k = 2*np.pi/wavelength
k_ix = k*np.sin(np.deg2rad(theta_i_deg))
k_mx = k_ix + m*2*np.pi/d

if abs(k_mx) <= k:
    print("No evanescent decay length for these settings.")
else:
    kappa_m = np.sqrt(k_mx**2-k**2)
    ell = 1/kappa_m
    multiples = np.arange(4)
    distances = multiples*ell
    amplitude_ratio = np.exp(-kappa_m*distances)
    print(f"Amplitude decay length = {ell:.4f} um")
    print("  s/ell     s (um)   amplitude ratio   squared ratio")
    print(np.round(np.column_stack(
        [multiples, distances, amplitude_ratio, amplitude_ratio**2]), 4))


<details><summary>Answer</summary>

For either order, the amplitude ratios at $0,1,2,3$ decay lengths are $1,0.3679,0.1353,0.0498$. The squared ratios are $1,0.1353,0.0183,0.0025$.

The physical distances differ: one decay length is approximately $0.2847\,\mu\mathrm m$ for $m=+1$ and $0.1389\,\mu\mathrm m$ for $m=+2$. The same ratio is reached sooner for the latter order. The squared-amplitude decay length is $\ell_m/2$.

</details>


## 6. Follow one order through grazing

Choose **normal incidence**, keep $\lambda=2\,\mu\mathrm m$, and follow **the same order $m=+1$** while changing $d$. Now

$$\frac{k_{mx}}k=\frac{\lambda}{d}.$$

| Period $d$ | $k_{mx}/k$ | State of $m=+1$ |
| --- | --- | --- |
| $3\,\mu\mathrm m$ | $2/3$ | Propagating: $\theta_m\approx41.81^\circ$ |
| $2\,\mu\mathrm m$ | $1$ | Grazing: $\theta_m=90^\circ$ |
| $1.5\,\mu\mathrm m$ | $4/3$ | Evanescent: no real outgoing angle |

The three buttons in the explorer reproduce these states. The figure below places them side by side with the **same physical window, time, colour scale and surface amplitude**.

**Predict:** should the field vanish at grazing? Should its amplitude already decay there?


In [ ]:
# Static comparison; it does not change the explorer controls.
comparison = evanescent_view.transition_figure()
try:
    display(Image(data=evanescent_view.figure_png(comparison)))
finally:
    comparison.clear()


<details><summary>Answer</summary>

At exact grazing, $k_{mz}=0$: the field varies with $x$ and time but has neither phase change nor amplitude decay with depth. Its phase travels parallel to the surface. With the prescribed $A=1$, it does not vanish.

On the propagating side of grazing, the real negative $k_{mz}$ approaches zero. On the evanescent side, $k_{mz}=-i\kappa_m$ and $\kappa_m$ grows from zero as the period is reduced farther. A changing propagation classification does not by itself specify the order's actual excitation amplitude.

</details>

### Small calculation 4: inspect both sides of the threshold

NumPy's complex square root lets us continue through grazing. We add `0j` before taking the square root, then take the **negative** branch for our outgoing side. This gives a negative real $k_{mz}$ for propagation and a negative imaginary $k_{mz}$ for evanescence.

**Predict and rerun:** compare $d=2.01$ with $d=1.99\,\mu\mathrm m$. Then replace the wavelength with $1.5\,\mu\mathrm m$: where is the new grazing threshold for $m=+1$ at normal incidence?


In [ ]:
wavelength = 2.0  # micrometres; repeat with 1.5
theta_i_deg, m = 0.0, 1
periods = np.array([3.0, 2.01, 2.0, 1.99, 1.5])
k = 2*np.pi/wavelength

print("d (um)   k_mx/k     k_mz/k             classification")
for d in periods:
    alpha = np.sin(np.deg2rad(theta_i_deg)) + m*wavelength/d
    r = 1-alpha**2
    if abs(r) <= 1e-12:
        r = 0.0
    k_mz_over_k = -np.sqrt(r+0j)
    state = "grazing" if r == 0 else ("propagating" if r > 0 else "evanescent")
    print(f"{d:5.2f}    {alpha:6.3f}    {k_mz_over_k.real:+.3f}{k_mz_over_k.imag:+.3f}j    {state}")


<details><summary>Answer and one more experiment</summary>

For $\lambda=2\,\mu\mathrm m$, $d=2.01$ is propagating and $d=1.99$ is evanescent; the exact threshold is $d=2$. For $\lambda=1.5\,\mu\mathrm m$, the threshold moves to $d=1.5$.

More generally, in this geometry the threshold condition is

$$\left|\sin\theta_i+m\frac{\lambda}{d}\right|=1.$$

At normal incidence and for a nonzero order, this becomes $d=|m|\lambda$. At tilted incidence the two signs are no longer symmetric.

**Check against the explorer:** restore **Notebook 3 example** ($\lambda=2$, $\theta_i=30^\circ$, $m=+1$), then increase $d$ to $4\,\mu\mathrm m$. Since $0.5+\lambda/d=1$, this is grazing. Increasing $d$ a little farther makes the selected order propagating. The integer $m$ never changed.

</details>


## 7. Does decay mean that energy is being absorbed?

Our surrounding medium is lossless. The exponential decrease here does **not** describe an outgoing beam gradually losing energy to absorption.

For a single evanescent electromagnetic component in a homogeneous lossless half-space, the time-averaged energy flow normal to the surface is zero; energy can flow parallel to it. The corresponding result is explained through total internal reflection in [TU Delft, “Total Internal Reflection and Evanescent Waves”](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html#total-internal-reflection-and-evanescent-waves). The grating selects the tangential wavevector differently, but the outgoing homogeneous medium supports the same type of decaying solution.

That is why this notebook labels $|U|^2/A^2$ as a **squared amplitude**, not as transmitted normal intensity. A field can be nonzero near the surface without carrying average power away as a beam.

“Candidate,” “excited,” and “propagating” answer different questions:

| Question | What supplies the answer? |
| --- | --- |
| Which horizontal components fit the repeated grating and its illumination? | The integer phase-repeat condition |
| Does one component propagate, graze, or decay in the outgoing medium? | The normal component from the wave equation |
| Is its amplitude nonzero, and how large is it? | The grating's scattering response, including its structure and illumination |

We prescribed the last answer to draw individual fields. An evanescent order is therefore **not automatically absent**, and a propagating order is **not automatically bright**.

These classifications refer to the ideal indefinitely periodic grating and the uniform outgoing half-space used here. The finite-source sums in [Spreading waves and finite arrays](../supplement/Spreading_Waves_and_Finite_Arrays.ipynb) address a different question: how interference peaks develop and acquire angular width.


## Final checkpoint

1. **Does an evanescent order violate the integer phase-repeat condition?**

   No. It fits that condition but has no real outgoing angle in the medium.

2. **What does the imaginary normal component do?**

   With the decaying branch for $z<0$, $k_{mz}=-i\kappa_m$ gives an amplitude factor $e^{-\kappa_m s}$, where $s=-z$.

3. **Does a white field snapshot at one point show that the envelope has vanished?**

   No. The signed field can be at an instantaneous zero while the envelope remains nonzero.

4. **What happens at grazing?**

   $k_{mz}=0$. There is no normal phase advance or decay; the phase travels along the surface.

5. **Does allowing an integer order tell us how strongly it is excited?**

   No. This notebook compares prescribed unit surface amplitudes. A scattering calculation supplies the actual amplitudes.

**What we have added:** the orders excluded from the earlier direction plots can have oscillating fields near the grating. For evanescent components, their normal dependence is exponential decay. As an evanescent order approaches grazing, its decay length grows without bound ($\ell_m\to\infty$). At exact grazing, this ideal component has no normal amplitude decay.

**Next: [Notebook 5: Grating diffraction in three dimensions](../5_Grating_Diffraction_in_Three_Dimensions.ipynb).** So far, $k_{iy}=k_{my}=0$: the phase is constant along each groove. Keep the same grating and outgoing $z<0$ side, but let the incident wave have $k_{iy}\ne0$. **How does phase variation along the grooves change which orders propagate and where they travel?**


## References and connections

- [TU Delft — Scalar Diffraction Optics, “Angular Spectrum Method”](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html#angular-spectrum-method): Eq. (375), the outgoing square-root branch, and the propagating/evanescent distinction. Its $+z$ outgoing convention is the reverse of this notebook's $-z$ convention; here $\lambda$ always means wavelength in the medium.
- [TU Delft — Basic Electromagnetic and Wave Optics, “Total Internal Reflection and Evanescent Waves”](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html#total-internal-reflection-and-evanescent-waves): decaying fields and energy flow parallel to an interface in a lossless medium.
- [Ansys Optics — Grating projections in FDTD overview, “Grating physics”](https://optics.ansys.com/hc/en-us/articles/360034394354-Grating-projections-in-FDTD-overview): discrete tangential components and the distinction between directions and scattering strengths.
- [Notebook 3 — Phase change and diffraction orders](../3_Why_Diffraction_Orders_Appear.ipynb): the origin of the integer condition used here.
- [From the wave equation to Harvey and Ewald](../supplement/n_Wave_Equation_Gratings_Harvey_and_Ewald.ipynb): a further geometrical treatment in three dimensions. That notebook uses an outgoing $+z$ convention.

To release the explorer's controls and stop playback when finished, run `evanescent_explorer.close()`. Rerunning its creation cell already does this before replacing it.
